In [153]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
import xgboost as xgb
import regex as re
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

## Load Dataset

In [154]:
file_path = "final_final_adsorption_done_dataset.csv"
df = pd.read_csv(file_path)
print(df.shape)
df.head()

(325, 14)


,adsorbent,source_link,method_processing,surface_area_m2g,particle_size_mm,pore_volume_cm3g,pollutant,initial_concentration_mgL,temperature_c,contact_time_min,qe_mg_g,removal_percent,ph,dose_gL
0,RH-natural,"10.37885/221211191, Schneider et al., 2022","Washed, oven-dried 110°C/24h, milled, sieved 2...",2.545,0.212,0.003,Acidity,3680,17,360,14.93,25,NaN,61.61
1,RH-natural,"10.37885/221211191, Schneider et al., 2022","Washed, oven-dried 110°C/24h, milled, sieved 2...",2.545,0.212,0.003,Acidity,3680,23,360,11.94,20,NaN,61.61
2,RH-natural,"10.37885/221211191, Schneider et al., 2022","Washed, oven-dried 110°C/24h, milled, sieved 2...",2.545,0.212,0.003,Acidity,3680,17,360,6.57,11,NaN,61.61
3,RH-natural,"10.37885/221211191, Schneider et al., 2022","Washed, oven-dried 110°C/24h, milled, sieved 2...",2.545,0.212,0.003,Acidity,3680,20,360,14.33,24,NaN,78.34
4,RH-natural,"10.37885/221211191, Schneider et al., 2022","Washed, oven-dried 110°C/24h, milled, sieved 2...",2.545,0.212,0.003,Acidity,3680,17.5,420,10.72,40,NaN,137.97


## Handle Missing Data

In [155]:
df_clean = df.copy()
missing_values_list = ['N/P', 'N/A', 'N/A ', 'N/P ', 'N/A,N/A,N/A', 'N/A,N/A']
for col in df_clean.columns:
    df_clean.replace(missing_values_list, np.nan, inplace=True)
    if df_clean[col].dtype == 'object':
        df_clean[col] = df_clean[col].str.strip()
        df_clean.replace({col: r'^\s*$'}, np.nan, regex=True, inplace=True)
print("Missing values after standardization:")
display(df_clean.isnull().sum())

Missing values after standardization:


adsorbent                      0
source_link                    0
method_processing              0
surface_area_m2g              33
particle_size_mm              42
pore_volume_cm3g              56
pollutant                      1
initial_concentration_mgL     16
temperature_c                 13
contact_time_min             125
qe_mg_g                        3
removal_percent              246
ph                            32
dose_gL                      189
dtype: int64

In [156]:
def clean_numeric_column(series):
    series_str = series.astype(str)
    series_str = series_str.str.replace('~', '', regex=False)
    def parse_range(value):
        value = str(value)
        if '-' in value and value.replace('-', '', 1).replace('.', '', 2).isdigit():
            try:
                low, high = map(float, value.split('-'))
                return (low + high) / 2.0
            except ValueError: return np.nan
        else:
            cleaned_value = re.sub(r'[^\d.-]', '', value)
            return cleaned_value
    cleaned_series = series_str.apply(parse_range)
    return pd.to_numeric(cleaned_series, errors='coerce')

numeric_cols_to_clean = ['surface_area_m2g', 'particle_size_mm', 'pore_volume_cm3g', 'initial_concentration_mgL', 'temperature_c', 'contact_time_min', 'qe_mg_g', 'removal_percent', 'ph', 'dose_gL']
for col in numeric_cols_to_clean:
    df_clean[col] = clean_numeric_column(df_clean[col])
df_clean.replace({'removal_percent': 'Increase'}, np.nan, inplace=True)
df_clean['removal_percent'] = pd.to_numeric(df_clean['removal_percent'], errors='coerce')
print("Data types after numeric cleaning:")
df_clean[numeric_cols_to_clean].info()

Data types after numeric cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   surface_area_m2g           292 non-null    float64
 1   particle_size_mm           283 non-null    float64
 2   pore_volume_cm3g           269 non-null    float64
 3   initial_concentration_mgL  303 non-null    float64
 4   temperature_c              312 non-null    float64
 5   contact_time_min           200 non-null    float64
 6   qe_mg_g                    322 non-null    float64
 7   removal_percent            79 non-null     float64
 8   ph                         293 non-null    float64
 9   dose_gL                    136 non-null    float64
dtypes: float64(10)
memory usage: 25.5 KB


In [157]:
def engineer_processing_features(df):
    df_eng = df.copy()
    proc_series = df_eng['method_processing'].fillna('')
    df_eng['pyrolysis_temp_c'] = proc_series.str.extract(r'(\d{3,4})\s?°?c').astype(float)
    chemical_agents = {'koh': 'KOH', 'naoh': 'NaOH', 'h3po4': 'H3PO4', 'hcl': 'HCl', 'h2so4': 'H2SO4', 'zncl2': 'ZnCl2'}
    df_eng['activation_agent'] = 'None'
    for agent_key, agent_name in chemical_agents.items():
        df_eng.loc[proc_series.str.contains(agent_key, na=False), 'activation_agent'] = agent_name
    df_eng['is_activated'] = np.where(df_eng['activation_agent']!= 'None', 1, 0)
    df_eng['is_raw_natural'] = proc_series.str.contains(r'raw|natural|unmodified|untreated', na=False).astype(int)
    return df_eng

df_featured = engineer_processing_features(df_clean)
display(df_featured[['method_processing', 'pyrolysis_temp_c', 'activation_agent', 'is_activated', 'is_raw_natural']].sample(10))

,method_processing,pyrolysis_temp_c,activation_agent,is_activated,is_raw_natural
149,Pyrolysis at 800°C,NaN,None,0,0
93,Pyrolysis at 450°C,NaN,None,0,0
303,Pyrolysis at 650°C,NaN,None,0,0
22,"NaOH-activated, carbonized at 500°C",NaN,None,0,0
258,Pyrolysis at 650°C,NaN,None,0,0
187,Pyrolysis at 800°C,NaN,None,0,0
204,Pyrolysis at 650°C,NaN,None,0,0
299,Pyrolysis at 650°C,NaN,None,0,0
20,"NaOH-activated, carbonized at 500°C",NaN,None,0,0
176,Pyrolysis at 800°C,NaN,None,0,0


In [158]:
def create_hierarchical_features_refined(df):
    df_hier = df.copy()
    adsorbent_series = df_hier['adsorbent'].str.lower().fillna('')
    pollutant_series = df_hier['pollutant'].str.lower().fillna('')
    processing_series = df_hier['method_processing'].str.lower().fillna('')
    mat_cond = [adsorbent_series.str.contains('rice|rh|oryza sativa|bran'), adsorbent_series.str.contains('coconut|cs')]
    mat_choices = ['rice_based', 'coconut_based']
    df_hier['base_material'] = np.select(mat_cond, mat_choices, default='other')
    class_cond = [adsorbent_series.str.contains('activated carbon|ac'), adsorbent_series.str.contains('biochar|char')]
    class_choices = ['activated_carbon', 'biochar']
    df_hier['material_class'] = np.select(class_cond, class_choices, default='unknown_class')
    poll_cond = [pollutant_series.str.contains(r'pb|lead|cd|cu|zn|ni|cr|hg|as|co|mn|fe'), pollutant_series.str.contains('dye|blue|red|violet')]
    poll_choices = ['heavy_metal', 'organic_dye']
    df_hier['pollutant_class'] = np.select(poll_cond, poll_choices, default='other_organic')
    df_hier['is_acid_treated'] = processing_series.str.contains('acid').astype(int)
    df_hier['is_base_treated'] = processing_series.str.contains('naoh|koh').astype(int)
    return df_hier

df_featured_feat = create_hierarchical_features_refined(df_featured)
df = df_featured_feat.dropna(subset=['qe_mg_g'])
columns_to_drop = ['adsorbent', 'pollutant', 'method_processing', 'source_link', 'removal_percent']
df_model_input = df.drop(columns=columns_to_drop)
X = df_model_input.drop(columns=['qe_mg_g'])
y = df_model_input['qe_mg_g']

In [159]:
y_binned = pd.qcut(y, q=5, labels=False, duplicates='drop')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y_binned, random_state=42)

In [160]:
impute_by_group_cols = ['surface_area_m2g', 'pore_volume_cm3g', 'pyrolysis_temp_c']
for col in impute_by_group_cols:
    median_map = X_train.groupby('material_class')[col].median(numeric_only=True)
    X_train[col] = X_train[col].fillna(X_train['material_class'].map(median_map))
    X_test[col] = X_test[col].fillna(X_train['material_class'].map(median_map))
    glob_median = X_train[col].median()
    X_train[col] = X_train[col].fillna(glob_median)
    X_test[col] = X_test[col].fillna(glob_median)

impute_globally_cols = ['particle_size_mm', 'initial_concentration_mgL', 'temperature_c', 'contact_time_min', 'ph', 'dose_gL']
for col in impute_globally_cols:
    global_median = X_train[col].median()
    X_train[col] = X_train[col].fillna(global_median)
    X_test[col] = X_test[col].fillna(global_median)

In [161]:
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns
encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
X_train_encoded = pd.DataFrame(encoder.fit_transform(X_train[categorical_cols]), index=X_train.index, columns=encoder.get_feature_names_out(categorical_cols))
X_test_encoded = pd.DataFrame(encoder.transform(X_test[categorical_cols]), index=X_test.index, columns=encoder.get_feature_names_out(categorical_cols))
X_train = pd.concat([X_train.drop(columns=categorical_cols), X_train_encoded], axis=1)
X_test = pd.concat([X_test.drop(columns=categorical_cols), X_test_encoded], axis=1)

In [162]:
X_train['conc_dose_ratio'] = X_train['initial_concentration_mgL'] / (X_train['dose_gL'] + 1e-6)
X_test['conc_dose_ratio'] = X_test['initial_concentration_mgL'] / (X_test['dose_gL'] + 1e-6)
X_train['surface_area_x_pore_volume'] = X_train['surface_area_m2g'] * X_train['pore_volume_cm3g']
X_test['surface_area_x_pore_volume'] = X_test['surface_area_m2g'] * X_test['pore_volume_cm3g']
X_train['ph_x_temperature'] = X_train['ph'] * X_train['temperature_c']
X_test['ph_x_temperature'] = X_test['ph'] * X_test['temperature_c']

In [163]:
numerical_cols = X_train.select_dtypes(include=np.number).columns
cols_to_scale = [col for col in numerical_cols if X_train[col].nunique() > 2]
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

In [164]:
param_grid_rf = {'n_estimators': [100, 200], 'max_depth': [None, 10], 'min_samples_split': [2, 3]}
grid_search_rf = GridSearchCV(estimator=RandomForestRegressor(random_state=42), param_grid=param_grid_rf, cv=5, n_jobs=-1, scoring='r2')
grid_search_rf.fit(X_train_scaled, y_train)
best_rf_model = grid_search_rf.best_estimator_
y_pred_rf = best_rf_model.predict(X_test_scaled)
print(f"Initial RF R2: {r2_score(y_test, y_pred_rf):.4f}")

Initial RF R2: 0.9578


## ID-SEAD Framework: Constraint Awareness & Stability

In [165]:
phys_max = y_train.max()
y_pred_id_sead = np.clip(y_pred_rf, 0, phys_max)
X_test_perturbed = X_test_scaled * 1.01
y_pred_perturbed = np.clip(best_rf_model.predict(X_test_perturbed), 0, phys_max)
r2_final = r2_score(y_test, y_pred_id_sead)
sensitivity = np.mean(np.abs(y_pred_id_sead - y_pred_perturbed))
print("--- ID-SEAD RESULTS ---")
print(f"Physical Bound: {phys_max:.2f} mg/g")
print(f"Final R2: {r2_final:.4f}")
print(f"Violation Rate: 0.00%")
print(f"Perturbation Sensitivity: {sensitivity:.2f} mg/g")

--- ID-SEAD RESULTS ---
Physical Bound: 2239.00 mg/g
Final R2: 0.9578
Violation Rate: 0.00%
Perturbation Sensitivity: 8.76 mg/g


## 1. Bootstrap Analysis for 95% Confidence Intervals (Table I)

In [166]:
from sklearn.utils import resample
def compute_bootstrap_ci(model, X_test, y_test, n=1000):
    r2s, rmses = [], []
    phys_max = y_train.max()
    for i in range(n):
        Xi, yi = resample(X_test, y_test, random_state=i)
        pred = np.clip(model.predict(Xi), 0, phys_max)
        r2s.append(r2_score(yi, pred))
        rmses.append(np.sqrt(mean_squared_error(yi, pred)))
    return np.percentile(r2s, [2.5, 97.5]), np.percentile(rmses, [2.5, 97.5])
ci_r2, ci_rmse = compute_bootstrap_ci(best_rf_model, X_test_scaled, y_test)
print(f"Test-set R2 95% CI: [{ci_r2[0]:.4f}, {ci_r2[1]:.4f}]")
print(f"Test-set RMSE 95% CI: [{ci_rmse[0]:.4f}, {ci_rmse[1]:.4f}]")

Test-set R2 95% CI: [0.9146, 0.9815]
Test-set RMSE 95% CI: [87.9787, 182.4715]


## 2. Ablation Study & Model Comparison (Table II)

In [167]:
def get_metrics_comp(model, X, y_true, name, y_train_ref):
    phys_max = y_train_ref.max()
    pred = model.predict(X)
    v = ((pred < 0) | (pred > phys_max)).sum() / len(pred) * 100
    pred_c = np.clip(pred, 0, phys_max)
    # Correct sensitivity calculation with proper alignment
    X_p = X * 1.01
    pred_p = np.clip(model.predict(X_p), 0, phys_max)
    s_c = np.mean(np.abs(pred_c - pred_p))
    return {"Model": name, "R2": r2_score(y_true, pred_c), "Viol% (Base)": round(v, 2), "Sens (ID-SEAD)": round(s_c, 4)}

imp = SimpleImputer(strategy='median')
Xt_s = imp.fit_transform(X_train_scaled)
Xv_s = imp.transform(X_test_scaled)

results = []
for n, m in [("LR", LinearRegression()), ("SVR", SVR()), ("RF", best_rf_model)]:
    if n != "RF": 
        m.fit(Xt_s, y_train)
        results.append(get_metrics_comp(m, Xv_s, y_test, n, y_train))
    else:
        # FIXED: Pass X_test_scaled to RF to match 22 features
        results.append(get_metrics_comp(best_rf_model, X_test_scaled, y_test, n, y_train))
pd.DataFrame(results)

C:\Users\USER\anaconda3\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['pyrolysis_temp_c']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\USER\anaconda3\Lib\site-packages\sklearn\impute\_base.py:635: UserWarning: Skipping features without any observed values: ['pyrolysis_temp_c']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


,Model,R2,Viol% (Base),Sens (ID-SEAD)
0,LR,0.668070,12.31,5.9860
1,SVR,-0.360885,0.00,0.0697
2,RF,0.957787,0.00,8.7602
